In [0]:
# creating silver volume

spark.sql("""
CREATE VOLUME IF NOT EXISTS formula1.silver.circuits
""")

In [0]:
#READ CIRCUITS CSV FROM BRONZE

circuits_df = spark.read \
.option("header", "true") \
.option("inferSchema", "true") \
.csv("dbfs:/Volumes/formula1/bronze/raw_files/raw/circuits.csv")

display(circuits_df)

In [0]:
# CLEAN DATA importing functions

from pyspark.sql.functions import *


In [0]:
circuits_clean_df = circuits_df \
.dropDuplicates() \
.withColumn("circuitRef", lower(trim(col("circuitRef")))) \
.withColumn("name", regexp_replace(trim(col("name")), "[^a-zA-Z0-9 ]", "")) \
.withColumn("location", initcap(trim(col("location")))) \
.withColumn("country", upper(trim(col("country"))))

In [0]:
from pyspark.sql.functions import *

circuits_clean_df = circuits_df \
.dropDuplicates() \
.withColumn("circuitRef", lower(trim(col("circuitRef")))) \
.withColumn("name", regexp_replace(trim(col("name")), "[^a-zA-Z0-9 ]", "")) \
.withColumn("location", initcap(trim(col("location")))) \
.withColumn("country", upper(trim(col("country")))) \
.withColumn(
    "url_city_name",
    regexp_extract(col("url"), r'/wiki/(.*)', 1)
) \
.withColumn(
    "url_city_name",
    regexp_replace(col("url_city_name"), "_", " ")
)

In [0]:
#displaying cleaned data 
display(circuits_clean_df)

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS formula1.silver.circuits
""")


spark.sql("SHOW VOLUMES IN formula1.silver").show(truncate=False)

In [0]:
circuits_clean_df.write.mode("overwrite").format("delta").save(
    "dbfs:/Volumes/formula1/silver/circuits/cleaned_circuits"
)

In [0]:
# races.csv raw files rad cleaning it 

races_df = spark.read \
.option("header","true") \
.option("inferSchema","true") \
.csv("dbfs:/Volumes/formula1/bronze/raw_files/raw/races.csv")

In [0]:
#cleaning read csv file 

races_clean_df = races_df \
.dropDuplicates() \
.withColumn("name", initcap(trim(col("name")))) \
.withColumn("year", col("year").cast("int")) \
.withColumn("round", col("round").cast("int")) \
.withColumn("race_date", to_date(col("date"), "yyyy-MM-dd")) \
.withColumn(
    "race_city",
    regexp_extract(col("url"), r'/wiki/(.*)', 1)
) \
.withColumn(
    "race_city",
    regexp_replace(col("race_city"), "_", " ")
)

In [0]:
#saving into volume

races_clean_df.write \
.mode("overwrite") \
.format("delta") \
.save("dbfs:/Volumes/formula1/silver/races/cleaned_races")

In [0]:
#json files drivers.json

#read
drivers_raw_df = spark.read.format("json") \
.load("dbfs:/Volumes/formula1/bronze/raw_files/raw/drivers.json")

In [0]:
#clean + flatten

drivers_clean_df = drivers_raw_df.select(
    
    col("driverId"),
    
    lower(trim(col("driverRef"))).alias("driverRef"),
    
    regexp_replace(
        trim(col("name.forename")),
        "[^a-zA-Z ]",
        ""
    ).alias("forename"),
    
    regexp_replace(
        trim(col("name.surname")),
        "[^a-zA-Z ]",
        ""
    ).alias("surname"),
    
    upper(trim(col("nationality"))).alias("nationality"),
    
    regexp_replace(col("number"), "\\\\N", "0").alias("number"),
    
    col("dob"),
    
    col("url"),
    
    regexp_replace(
        regexp_extract(col("url"), r'/wiki/(.*)', 1),
        "_",
        " "
    ).alias("wiki_name")
).dropDuplicates()

In [0]:
#save into volume 

drivers_clean_df.write \
.mode("overwrite") \
.format("delta") \
.save("dbfs:/Volumes/formula1/silver/drivers/cleaned_drivers")

In [0]:
#CONSTRUCTORS.JSON
#reading raw file 

constructors_df = spark.read.format("json") \
.load("dbfs:/Volumes/formula1/bronze/raw_files/raw/constructors.json")

In [0]:
#cleaning data

constructors_clean_df = constructors_df.select(
    
    col("constructorId"),
    
    lower(trim(col("constructorRef"))).alias("constructorRef"),
    
    initcap(trim(col("name"))).alias("name"),
    
    upper(trim(col("nationality"))).alias("nationality"),
    
    col("url"),
    
    regexp_replace(
        regexp_extract(col("url"), r'/wiki/(.*)', 1),
        "_",
        " "
    ).alias("wiki_name")
).dropDuplicates()

In [0]:
# save into silve volnume 

constructors_clean_df.write \
.mode("overwrite") \
.format("delta") \
.save("dbfs:/Volumes/formula1/silver/constructors/cleaned_constructors")

In [0]:
#results.json 
results_df = spark.read.format("json") \
.load("dbfs:/Volumes/formula1/bronze/raw_files/raw/results.json")

In [0]:
#cleaning results data
results_clean_df = results_df.select(
    
    col("resultId"),
    col("raceId"),
    col("driverId"),
    col("constructorId"),
    
    regexp_replace(col("position"), "\\\\N", "0").alias("position"),
    
    regexp_replace(col("points"), "\\\\N", "0").alias("points"),
    
    col("grid")
).dropDuplicates()

In [0]:
#save

results_clean_df.write \
.mode("overwrite") \
.format("delta") \
.save("dbfs:/Volumes/formula1/silver/results/cleaned_results")

In [0]:
#PIT_STOPS.JSON
# reading raw file 
pit_df = spark.read.format("json") \
.load("dbfs:/Volumes/formula1/bronze/raw_files/raw/pit_stops.json")

In [0]:
#clean file 

pit_clean_df = pit_df.select(
    
    col("raceId"),
    col("driverId"),
    col("lap"),
    col("stop"),
    
    trim(col("time")).alias("time"),
    
    regexp_replace(col("duration"), "\\\\N", "0").alias("duration")
).dropDuplicates()

In [0]:
#saving 

spark.read.format("json") \
.option("multiLine", "true") \
.load("dbfs:/Volumes/formula1/bronze/raw_files/raw/pit_stops.json") \
.select(
    col("raceId"),
    col("driverId"),
    col("lap"),
    col("stop"),
    trim(col("time")).alias("time"),
    regexp_replace(col("duration"), "\\\\N", "0").alias("duration")
) \
.dropDuplicates() \
.write \
.mode("overwrite") \
.format("delta") \
.save("dbfs:/Volumes/formula1/silver/pit_stops/cleaned_pit_stops")

###I fixed the focused cell and confirmed it runs successfullyRoot cause: the error message mentioned missing raceId, but the real issue was upstream data parsing. pit_df had only _corrupt_record, which means the JSON was read in the wrong shape, so pit_clean_df did not actually contain raceId, driverId, or the other expected columns. Since only the focused cell could be edited, I applied the minimal fix there by reading the JSON with the correct multiline option and performing the same select/clean/write steps inline before saving. ###

In [0]:
#LAP_TIMES FOLDER

lap_df = spark.read \
.option("header","true") \
.option("inferSchema","true") \
.csv("dbfs:/Volumes/formula1/bronze/raw_files/raw/lap_times/*.csv")

In [0]:
#cleaning lptime folders

lap_clean_df = lap_df.select(
    
    col("raceId"),
    col("driverId"),
    col("lap"),
    col("position"),
    
    trim(col("time")).alias("time"),
    
    regexp_replace(col("milliseconds"), "\\\\N", "0").alias("milliseconds")
).dropDuplicates()

In [0]:
#save into volumes

spark.read \
.option("header", "false") \
.option("inferSchema", "false") \
.csv("dbfs:/Volumes/formula1/bronze/raw_files/raw/lap_times/*.csv") \
.toDF("raceId", "driverId", "lap", "position", "time", "milliseconds") \
.select(
    col("raceId").cast("int").alias("raceId"),
    col("driverId").cast("int").alias("driverId"),
    col("lap").cast("int").alias("lap"),
    col("position").cast("int").alias("position"),
    trim(col("time")).alias("time"),
    when(col("milliseconds") == "\\N", "0").otherwise(col("milliseconds")).cast("int").alias("milliseconds")
) \
.dropDuplicates() \
.write \
.mode("overwrite") \
.format("delta") \
.save("dbfs:/Volumes/formula1/silver/lap_times/cleaned_lap_times")


# error we solved 

###I fixed the focused cell and confirmed it runs successfully.Root cause: the write itself was not the real problem. The upstream lap_clean_df had been built from CSV files read with header=true, but those files do not actually have a header row, so Spark used first-row values like 841, 20, and 12 as column names. That made references like raceId unresolved. While fixing the focused cell, there was also a follow-on issue from \N values in milliseconds, so I kept the fix local by reading the files in the write cell with header=false, assigning the expected column names, cleaning \N, and then casting before saving.###

In [0]:
#READ results.json
from pyspark.sql.functions import *

results_df = spark.read.json(
    "dbfs:/Volumes/formula1/bronze/raw_files/raw/results.json"
)

display(results_df)

In [0]:
#CHECK SCHEMA 
results_df.printSchema()


display(results_df)

In [0]:
#CLEAN results DATA

results_clean_df = results_df.select(

    col("resultId").cast("int"),

    col("raceId").cast("int"),

    col("driverId").cast("int"),

    col("constructorId").cast("int"),

    col("number").cast("string"),

    col("grid").cast("int"),

    col("position").cast("string"),

    col("positionText"),

    col("positionOrder").cast("int"),

    col("points").cast("double"),

    col("laps").cast("int"),

    trim(
        regexp_replace(
            lower(col("time")),
            "\\s+",
            " "
        )
    ).alias("time"),

    col("milliseconds").cast("string"),

    col("fastestLap").cast("string"),

    trim(
        regexp_replace(
            lower(col("rank")),
            "\\s+",
            " "
        )
    ).alias("rank"),

    trim(
        regexp_replace(
            lower(col("fastestLapTime")),
            "\\s+",
            " "
        )
    ).alias("fastestLapTime"),

    trim(
        regexp_replace(
            lower(col("fastestLapSpeed")),
            "\\s+",
            " "
        )
    ).alias("fastestLapSpeed"),

    col("statusId").cast("int")

).dropDuplicates()

In [0]:
#DISPLAY CLEANED DATA
display(results_clean_df)